# Pothole Detection | Hybrid Models
**Models:** YOLOv8m+CBAM, YOLOv8m+CoordAttn, YOLO+FasterRCNN Ensemble 

In [1]:
# Frees GPU memory by deleting any model objects left in the namespace from a previous run and emptying the CUDA cache.
import torch, gc

# Delete any existing models in memory
for var in [
    "cbam_yolo",
    "ca_yolo",
    "yolo_ens",
    "frcnn_ens",
    "cbam_injector",
    "ca_injector",
    "test_model",
    "test_model_cbam",
    "test_model_ca",
]:
    if var in dir():
        del var

# Clear GPU cache
torch.cuda.empty_cache()
gc.collect()

# Check memory after clearing
print("GPU Memory after clearing:")
print(f"  Allocated : {torch.cuda.memory_allocated() / 1024**2:.1f} MB")
print(f"  Reserved  : {torch.cuda.memory_reserved() / 1024**2:.1f} MB")
print(
    f"  Free      : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**2:.1f} MB"
)
print(
    f"  Total     : {torch.cuda.get_device_properties(0).total_memory / 1024**2:.1f} MB"
)

GPU Memory after clearing:
  Allocated : 0.0 MB
  Reserved  : 0.0 MB
  Free      : 24251.6 MB
  Total     : 24251.6 MB


In [2]:
2  # Prints the GPU name to confirm which device the notebook is running on.
import torch

print(f"   GPU: {torch.cuda.get_device_name(0)}")

   GPU: NVIDIA GeForce RTX 3090


In [4]:
# Detects whether the notebook is running on Kaggle, Colab, or local Jupyter and sets ROOT, SAVE_DIR, HYBRID_DIR and OUTPUT_DIR accordingly.
import os, sys

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
ON_JUPYTER = not ON_KAGGLE and not ON_COLAB

if ON_KAGGLE:
    print("Running on KAGGLE")
    ROOT = "/kaggle/working"
    SAVE_DIR = "/kaggle/working/saved_models"
elif ON_COLAB:
    print("Running on GOOGLE COLAB")
    ROOT = "/content"
    SAVE_DIR = "/content/saved_models"
else:
    print("Running on LOCAL JUPYTER")
    ROOT = "."
    SAVE_DIR = "./saved_models"

OUTPUT_DIR = os.path.join(ROOT, "hybrid_results")
DATA_DIR = os.path.join(ROOT, "data")
HYBRID_DIR = os.path.join(ROOT, "hybrid_models")

for d in [OUTPUT_DIR, DATA_DIR, HYBRID_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"   Output       -> {OUTPUT_DIR}")
print(f"   Base Weights -> {SAVE_DIR}")
print(f"   Hybrid Saves -> {HYBRID_DIR}")

Running on LOCAL JUPYTER
   Output       -> ./hybrid_results
   Base Weights -> ./saved_models
   Hybrid Saves -> ./hybrid_models


In [5]:
# Loads all shared imports and defines the fixed evaluation config, DEVICE, CONF_THRESH, IOU_THRESH, IMG_SIZE, and the chart palette.
import time, json, glob, warnings, xml.etree.ElementTree as ET
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONF_THRESH = 0.25
IOU_THRESH = 0.45
IMG_SIZE = 640
MAX_IMAGES = 150
NUM_WORKERS = (
    2  # cluster-safe Ultralytics/torchvision defaults (8) can exceed cluster ulimits
)


EPOCHS_FROZEN = 10  # phase 1: backbone frozen, attention learns fast
LR_FROZEN = 1e-3
EPOCHS_FULL = 40  # phase 2: everything trains together, gently
LR_FULL = 2e-4
WEIGHT_DECAY = 5e-4

# Confidence grid for the F1 operating-point sweep
CONF_SWEEP = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
]

# Paths to our fine-tuned base weights
YOLOV8M_WEIGHTS = f"{SAVE_DIR}/YOLOv8m_finetuned.pt"
FASTERRCNN_WEIGHTS = f"{SAVE_DIR}/faster_rcnn_finetuned.pth"

PALETTE = {
    "YOLOv8m+CBAM": "#f59e0b",
    "YOLOv8m+CoordAttn": "#10b981",
    "YOLO+FRCNN Ensemble": "#8b5cf6",
    "YOLOv8m (base)": "#00d4ff",
    "Faster R-CNN (base)": "#f97316",
}

print(f" Config loaded  |  Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(
    f"   Two-phase: {EPOCHS_FROZEN}ep @ lr={LR_FROZEN} (freeze=10) "
    f"-> {EPOCHS_FULL}ep @ lr={LR_FULL} (freeze=0)"
)

# Quick weight file check
for name, path in [("YOLOv8m", YOLOV8M_WEIGHTS), ("Faster R-CNN", FASTERRCNN_WEIGHTS)]:
    exists = os.path.exists(path)
    print(f"   {name} weights: {'FOUND' if exists else 'NOT FOUND'} at {path}")


 Config loaded  |  Device: cuda
   GPU: NVIDIA GeForce RTX 3090
   Two-phase: 10ep @ lr=0.001 (freeze=10) -> 40ep @ lr=0.0002 (freeze=0)
   YOLOv8m weights: FOUND at ./saved_models/YOLOv8m_finetuned.pt
   Faster R-CNN weights: FOUND at ./saved_models/faster_rcnn_finetuned.pth


In [6]:
# Downloads the three Kaggle datasets, loads them through the unified annotation loader, deduplicates, and defines the canonical train/val split shared with the other notebooks.
import kagglehub
from pathlib import Path

print(" Downloading datasets via kagglehub ...")
path_1 = kagglehub.dataset_download("chitholian/annotated-potholes-dataset")
path_2 = kagglehub.dataset_download("andrewmvd/pothole-detection")
path_3 = kagglehub.dataset_download("ashishkumarak/training-setzip")

DATASET_ROOTS = {"chitholian": path_1, "andrewmvd": path_2, "ashishkumar": path_3}
print("Datasets ready")


import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from PIL import Image


def load_annotated_potholes(root):
    root = Path(root)
    records = []
    for img_path in list(root.rglob("*.jpg")) + list(root.rglob("*.png")):
        xml_path = img_path.with_suffix(".xml")
        if not xml_path.exists():
            xml_path = img_path.parent.parent / "annotations" / (img_path.stem + ".xml")
        gt_boxes = []
        if xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                for obj in tree.findall("object"):
                    bb = obj.find("bndbox")
                    gt_boxes.append(
                        {
                            "label": (obj.find("name").text or "pothole").lower(),
                            "xmin": float(bb.find("xmin").text),
                            "ymin": float(bb.find("ymin").text),
                            "xmax": float(bb.find("xmax").text),
                            "ymax": float(bb.find("ymax").text),
                        }
                    )
            except Exception:
                pass
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


def load_ashishkumar_csv(root):
    root = Path(root)
    csv_path = root / "train" / "labels.csv"
    img_dir = root / "train" / "images"
    df = pd.read_csv(csv_path)
    grouped = df.groupby("ImageID")

    records = []
    for img_path in sorted(img_dir.glob("*.jpg")):
        gt_boxes = []
        if img_path.name in grouped.groups:
            for _, row in grouped.get_group(img_path.name).iterrows():
                gt_boxes.append(
                    {
                        "label": "pothole",
                        "xmin": float(row["XMin"]),
                        "ymin": float(row["YMin"]),
                        "xmax": float(row["XMax"]),
                        "ymax": float(row["YMax"]),
                    }
                )
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


all_records = []
for name, root in DATASET_ROOTS.items():
    if name == "ashishkumar":
        recs = load_ashishkumar_csv(root)
    else:
        recs = load_annotated_potholes(root)
    print(
        f"  {name}: {len(recs)} images ({sum(len(r['gt_boxes']) for r in recs)} gt boxes)"
    )
    all_records.extend(recs)

print(f"\nTotal images before dedup: {len(all_records)}")

# Normalized-pixel nearest-neighbor dedup (robust to re-encoding differences)
NORM_SIZE = (64, 64)
DEDUP_THRESHOLD = 1.0  # mean abs pixel diff (0-255 scale), true duplicates
# measured at 0.10-0.50, unrelated images much higher


def normalized_pixels(path):
    with Image.open(path) as img:
        return np.asarray(
            img.convert("L").resize(NORM_SIZE, Image.LANCZOS), dtype=np.float32
        ).ravel()


annotated_only = [r for r in all_records if r["gt_boxes"]]
print("Computing normalized pixel arrays for dedup ...")
all_arrs = np.stack([normalized_pixels(r["image_path"]) for r in annotated_only])

keep_mask = np.ones(len(annotated_only), dtype=bool)
seen_arrs = []
for i in range(len(annotated_only)):
    if not keep_mask[i]:
        continue
    if seen_arrs:
        diffs = np.abs(np.stack(seen_arrs) - all_arrs[i]).mean(axis=1)
        if diffs.min() < DEDUP_THRESHOLD:
            keep_mask[i] = False
            continue
    seen_arrs.append(all_arrs[i])

records = [r for r, keep in zip(annotated_only, keep_mask) if keep]

n_before = len(all_records)
n_after = len(records)
print(f"Total images after dedup: {n_after}")
print(f"   Duplicates removed: {n_before - n_after}")


import random as _random

_random.seed(42)
_shuffled = records.copy()
_random.shuffle(_shuffled)
_split_idx = int(len(_shuffled) * 0.8)
train_recs = _shuffled[:_split_idx]
val_recs = _shuffled[_split_idx:]
print(
    f"\nCanonical split: train={len(train_recs)}  val={len(val_recs)}  "
    f"(seed=42, shared by all training and evaluation cells below)"
)


Datasets ready
  chitholian: 665 images (1740 gt boxes)
  andrewmvd: 665 images (1740 gt boxes)
  ashishkumar: 674 images (1371 gt boxes)

Total images before dedup: 2004
Computing normalized pixel arrays for dedup ...
Total images after dedup: 926
   Duplicates removed: 1078

Canonical split: train=740  val=186  (seed=42, shared by all training and evaluation cells below)



## Fine-tune YOLOv8m baseline

In [7]:
# Fine-tunes the YOLOv8m baseline that serves as the ensemble's YOLO member, converting the canonical split to YOLO format first and skipping if the weights already exist.
import shutil, yaml, gc
from ultralytics import YOLO

YOLO_DIR = os.path.join(ROOT, "yolo_dataset")
DATA_YAML = f"{YOLO_DIR}/data.yaml"


def ensure_yolo_dataset():
    """Build the YOLO-format dataset from `records` if not already present.
    Idempotent safe to call multiple times or after a job restart."""
    if os.path.exists(DATA_YAML):
        print(f"data.yaml found -> {DATA_YAML}")
        return
    print("WARNING: yolo_dataset not found -- creating it now ...")
    for split in ["images/train", "images/val", "labels/train", "labels/val"]:
        os.makedirs(f"{YOLO_DIR}/{split}", exist_ok=True)

    def convert_to_yolo(rec_list, split):
        for rec in rec_list:
            img = cv2.imread(str(rec["image_path"]))
            if img is None:
                continue
            h, w = img.shape[:2]
            dst = f"{YOLO_DIR}/images/{split}/{rec['image_path'].name}"
            if not os.path.exists(dst):
                shutil.copy(str(rec["image_path"]), dst)
            with open(
                f"{YOLO_DIR}/labels/{split}/{rec['image_path'].stem}.txt", "w"
            ) as f:
                for g in rec["gt_boxes"]:
                    cx = ((g["xmin"] + g["xmax"]) / 2) / w
                    cy = ((g["ymin"] + g["ymax"]) / 2) / h
                    bw = (g["xmax"] - g["xmin"]) / w
                    bh = (g["ymax"] - g["ymin"]) / h
                    f.write(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

    convert_to_yolo(train_recs, "train")
    convert_to_yolo(val_recs, "val")
    with open(DATA_YAML, "w") as f:
        yaml.dump(
            {
                "path": YOLO_DIR,
                "train": "images/train",
                "val": "images/val",
                "nc": 1,
                "names": ["pothole"],
            },
            f,
        )
    print(f" data.yaml created at {DATA_YAML}")


if os.path.exists(YOLOV8M_WEIGHTS):
    print(f" YOLOv8m weights already exist at {YOLOV8M_WEIGHTS}  (skipping training)")
else:
    ensure_yolo_dataset()

    print("\n Fine-tuning YOLOv8m baseline ...")
    yolo_base = YOLO("yolov8m.pt")
    yolo_base.to(DEVICE)

    try:
        yolo_base.train(
            data=DATA_YAML,
            epochs=50,
            imgsz=IMG_SIZE,
            batch=16,
            lr0=1e-3,
            device=DEVICE,
            name="YOLOv8m_baseline_hybrid",
            exist_ok=True,  #  allows safe resume/re-run without name clashes
            workers=NUM_WORKERS,  #  cluster-safe worker count
            verbose=False,
            plots=False,
            save=True,
        )
    except Exception as e:
        print(f" Training failed: {e}")
        raise
    finally:
        # Always free GPU memory, even if training raised
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    best = "runs/detect/YOLOv8m_baseline_hybrid/weights/best.pt"
    last = "runs/detect/YOLOv8m_baseline_hybrid/weights/last.pt"
    src_weights = best if os.path.exists(best) else last
    os.makedirs(SAVE_DIR, exist_ok=True)
    shutil.copy(src_weights, YOLOV8M_WEIGHTS)
    print(f" YOLOv8m baseline saved -> {YOLOV8M_WEIGHTS}")

    del yolo_base
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


 YOLOv8m weights already exist at ./saved_models/YOLOv8m_finetuned.pt  (skipping training)


---
## Fine-tune Faster R-CNN baseline (self-contained, cluster-safe)

torchvision has no built-in `.train()` like Ultralytics, so this is a
manual training loop. Cluster-safety measures: `num_workers=2` (not the
often-higher framework default), per-epoch checkpointing to disk so a
killed/resumed cluster job doesn't lose progress, a `try/except/finally`
around the training loop so GPU memory is always freed even on failure,
and automatic skip if a final checkpoint already exists.

In [8]:
# Fine-tunes the Faster R-CNN baseline that serves as the ensemble's second member, with a manual training loop, per-epoch checkpointing, and automatic skip if a final checkpoint exists.
import gc, json as _json
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

FRCNN_EPOCHS = 20
FRCNN_LR = 5e-4
FRCNN_BATCH = 4  # Faster R-CNN is far more memory-hungry per-image than YOLO
FRCNN_CKPT_DIR = os.path.join(HYBRID_DIR, "frcnn_checkpoints")
os.makedirs(FRCNN_CKPT_DIR, exist_ok=True)


def _valid_box(g, min_size=1.0):
    """FIX: torchvision's Faster R-CNN hard-crashes (AssertionError, not a
    catchable-per-batch warning) on any box with zero or negative width/
    height. A single bad annotation anywhere in the dataset kills the
    entire training run. Filter degenerate boxes defensively min_size=1.0
    also drops sub-pixel boxes that are technically positive-area but
    numerically risky."""
    w = g["xmax"] - g["xmin"]
    h = g["ymax"] - g["ymin"]
    return w >= min_size and h >= min_size


class PotholeDatasetFRCNN(Dataset):
    """Wraps `records` (image_path + gt_boxes dicts) for torchvision detection."""

    def __init__(self, recs):
        # Filter degenerate boxes per-record at construction time, and
        # drop any record left with zero valid boxes afterward (rather than
        # only checking `if r['gt_boxes']`, which doesn't catch a record
        # whose boxes are all degenerate).
        self.recs = []
        n_dropped_boxes = 0
        n_dropped_images = 0
        for r in recs:
            valid_boxes = [g for g in r["gt_boxes"] if _valid_box(g)]
            n_dropped_boxes += len(r["gt_boxes"]) - len(valid_boxes)
            if valid_boxes:
                self.recs.append({**r, "gt_boxes": valid_boxes})
            elif r["gt_boxes"]:
                n_dropped_images += 1
        if n_dropped_boxes or n_dropped_images:
            print(
                f" Dropped {n_dropped_boxes} degenerate box(es) "
                f"({n_dropped_images} image(s) left with no valid boxes, excluded)"
            )

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, idx):
        rec = self.recs[idx]
        img = cv2.imread(str(rec["image_path"]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        boxes, labels = [], []
        for g in rec["gt_boxes"]:
            boxes.append([g["xmin"], g["ymin"], g["xmax"], g["ymax"]])
            labels.append(1)  # single class: pothole

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
        }
        return img_t, target


def frcnn_collate_fn(batch):
    return tuple(zip(*batch))


if os.path.exists(FASTERRCNN_WEIGHTS):
    print(
        f"Faster R-CNN weights already exist at {FASTERRCNN_WEIGHTS}  (skipping training)"
    )
else:
    print("\nFine-tuning Faster R-CNN baseline ...")

    train_ds = PotholeDatasetFRCNN(train_recs)
    train_loader = DataLoader(
        train_ds,
        batch_size=FRCNN_BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,  # cluster-safe, not torchvision's higher default
        collate_fn=frcnn_collate_fn,
    )

    frcnn_model = fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
    in_feat = frcnn_model.roi_heads.box_predictor.cls_score.in_features
    frcnn_model.roi_heads.box_predictor = FastRCNNPredictor(
        in_feat, num_classes=2
    )  # background + pothole
    frcnn_model.to(DEVICE)

    params = [p for p in frcnn_model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=FRCNN_LR, momentum=0.9, weight_decay=5e-4)

    start_epoch = 0
    ckpt_files = sorted(glob.glob(os.path.join(FRCNN_CKPT_DIR, "epoch_*.pt")))
    if ckpt_files:
        latest = ckpt_files[-1]
        print(f"   Resuming from checkpoint: {latest}")
        ckpt = torch.load(latest, map_location=DEVICE)
        frcnn_model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = ckpt["epoch"] + 1

    try:
        for epoch in range(start_epoch, FRCNN_EPOCHS):
            frcnn_model.train()
            epoch_loss = 0.0
            n_batches = 0
            for imgs, targets in tqdm(
                train_loader, desc=f"  Epoch {epoch + 1}/{FRCNN_EPOCHS}"
            ):
                imgs = [img.to(DEVICE) for img in imgs]
                targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

                loss_dict = frcnn_model(imgs, targets)
                loss = sum(loss_dict.values())

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                n_batches += 1

            avg_loss = epoch_loss / max(n_batches, 1)
            print(f"  Epoch {epoch + 1}/{FRCNN_EPOCHS}  avg_loss={avg_loss:.4f}")

            # checkpoint every epoch cheap insurance against cluster
            # job preemption / walltime limits killing a multi-hour run
            ckpt_path = os.path.join(FRCNN_CKPT_DIR, f"epoch_{epoch:03d}.pt")
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": frcnn_model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "avg_loss": avg_loss,
                },
                ckpt_path,
            )

    except Exception as e:
        print(f"Training failed at epoch {epoch}: {e}")
        raise
    finally:
        # Always free GPU memory, even if training raised or was interrupted
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    os.makedirs(SAVE_DIR, exist_ok=True)
    torch.save(frcnn_model.state_dict(), FASTERRCNN_WEIGHTS)
    print(f"Faster R-CNN baseline saved at {FASTERRCNN_WEIGHTS}")

    # Clean up per-epoch checkpoints now that final weights are saved
    for f in glob.glob(os.path.join(FRCNN_CKPT_DIR, "epoch_*.pt")):
        os.remove(f)

    del frcnn_model, optimizer, train_loader, train_ds
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


Faster R-CNN weights already exist at ./saved_models/faster_rcnn_finetuned.pth  (skipping training)


In [9]:
# Defines box_iou, compute_ap, evaluate, the confidence sweep and the throughput helper, the shared scoring functions used for every model in this notebook.


def box_iou(b1, b2):
    ix1 = max(b1[0], b2[0])
    iy1 = max(b1[1], b2[1])
    ix2 = min(b1[2], b2[2])
    iy2 = min(b1[3], b2[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    u = a1 + a2 - inter
    return inter / u if u > 0 else 0.0


def compute_ap_voc11(recalls, precisions):
    """PASCAL VOC 2007 11-point interpolated AP"""
    ap = 0.0
    for thr in np.linspace(0, 1, 11):
        ps = [p for r, p in zip(recalls, precisions) if r >= thr]
        ap += max(ps) if ps else 0.0
    return ap / 11


def compute_ap(recalls, precisions):
    # COCO-style 101-point interpolated AP.
    mrec = np.concatenate(([0.0], np.asarray(recalls, dtype=float), [1.0]))
    mpre = np.concatenate(([1.0], np.asarray(precisions, dtype=float), [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))  # precision envelope
    x = np.linspace(0, 1, 101)
    _trapz = getattr(np, "trapezoid", None) or np.trapz  # numpy>=2 rename
    return float(_trapz(np.interp(x, mrec, mpre), x))


def evaluate(all_gt, all_preds, iou_thr=0.5):
    # Greedy score-ordered matching, one detection per ground-truth box.
    pl = []
    for idx, preds in enumerate(all_preds):
        for p in preds:
            pl.append((idx, p["conf"], [p["xmin"], p["ymin"], p["xmax"], p["ymax"]]))
    pl.sort(key=lambda x: -x[1])

    num_gt = sum(len(g) for g in all_gt)
    matched = defaultdict(set)
    TP, FP = [], []
    for idx, conf, pb in pl:
        gts = all_gt[idx]
        taken = matched[idx]
        best_iou, best_j = 0.0, -1
        for j, g in enumerate(gts):
            if j in taken:  # <-- the fix
                continue
            iou = box_iou(pb, [g["xmin"], g["ymin"], g["xmax"], g["ymax"]])
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_thr and best_j >= 0:
            TP.append(1)
            FP.append(0)
            taken.add(best_j)
        else:
            TP.append(0)
            FP.append(1)

    tp_c = np.cumsum(TP)
    fp_c = np.cumsum(FP)
    recs = (tp_c / num_gt).tolist() if num_gt > 0 else [0.0]
    precs = (tp_c / np.maximum(tp_c + fp_c, 1e-12)).tolist()

    if not pl or num_gt == 0:
        ap_coco = ap_voc = 0.0
    else:
        ap_coco = compute_ap(recs, precs)
        ap_voc = compute_ap_voc11(recs, precs)

    ttp = int(sum(TP))
    tfp = int(sum(FP))
    tfn = num_gt - ttp
    pr = ttp / (ttp + tfp) if (ttp + tfp) > 0 else 0.0
    rc = ttp / (ttp + tfn) if (ttp + tfn) > 0 else 0.0
    f1 = 2 * pr * rc / (pr + rc) if (pr + rc) > 0 else 0.0
    return {
        "mAP@0.5": round(ap_coco, 4),
        "mAP@0.5(VOC11)": round(ap_voc, 4),
        "Precision": round(pr, 4),
        "Recall": round(rc, 4),
        "F1": round(f1, 4),
        "TP": ttp,
        "FP": tfp,
        "FN": tfn,
    }


# FPS measurement
FPS_WARMUP = 5


def _sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def fps_from_times(times, warmup=FPS_WARMUP):
    """Throughput in images/sec from a list of per-image durations."""
    t = [x for x in times[warmup:] if x > 0]
    if not t:
        t = [x for x in times if x > 0]
    return round(len(t) / sum(t), 1) if t else 0.0


def threshold_preds(all_preds, conf):
    return [[p for p in preds if p["conf"] >= conf] for preds in all_preds]


def sweep_best_conf(all_gt, all_preds, conf_grid=None, iou_thr=0.5, verbose=True):
    """Return (best_conf, metrics_at_best, full_sweep_dataframe).

    Sweeps the confidence threshold and picks the point that maximises F1,
    matching the operating point Ultralytics' validator reports.
    """
    conf_grid = conf_grid if conf_grid is not None else CONF_SWEEP
    rows = []
    for c in conf_grid:
        m = evaluate(all_gt, threshold_preds(all_preds, c), iou_thr=iou_thr)
        rows.append(
            {
                "conf": c,
                **{k: m[k] for k in ("Precision", "Recall", "F1", "TP", "FP", "FN")},
            }
        )
    df_sweep = pd.DataFrame(rows).set_index("conf")
    best_conf = float(df_sweep["F1"].idxmax())
    best_m = evaluate(all_gt, threshold_preds(all_preds, best_conf), iou_thr=iou_thr)
    if verbose:
        print(
            f"     conf sweep -> best F1={best_m['F1']:.4f} at conf={best_conf:.2f} "
            f"(P={best_m['Precision']:.4f}  R={best_m['Recall']:.4f})"
        )
    return best_conf, best_m, df_sweep


def summarize(
    all_gt, all_preds, fps_times=None, conf_grid=None, iou_thr=0.5, verbose=True
):
    """Build the standard result dict from UNTHRESHOLDED predictions.

    `all_preds` must come from a near-zero conf inference run (conf=0.001),
    so that the mAP integration and the sweep both see the full PR curve.
    """
    m_map = evaluate(all_gt, all_preds, iou_thr=iou_thr)  # full curve -> mAP
    m_def = evaluate(all_gt, threshold_preds(all_preds, CONF_THRESH), iou_thr=iou_thr)
    best_conf, m_best, df_sweep = sweep_best_conf(
        all_gt, all_preds, conf_grid=conf_grid, iou_thr=iou_thr, verbose=verbose
    )

    out = {
        "mAP@0.5": m_map["mAP@0.5"],
        "Precision": m_def["Precision"],
        "Recall": m_def["Recall"],
        "F1": m_def["F1"],
        "TP": m_def["TP"],
        "FP": m_def["FP"],
        "FN": m_def["FN"],
        "best_conf": best_conf,
        "P@best": m_best["Precision"],
        "R@best": m_best["Recall"],
        "F1@best": m_best["F1"],
    }
    if fps_times:
        out["FPS"] = fps_from_times(fps_times)
    out["_sweep"] = df_sweep
    return out


WARMUP_ITERS = 10


def warmup(forward, n=WARMUP_ITERS):
    """Run `forward` n times untimed, then synchronise."""
    for _ in range(n):
        forward()
    _sync()


def first_image(recs):
    """First decodable image in `recs`, or None."""
    for rec in recs:
        img = cv2.imread(str(rec["image_path"]))
        if img is not None:
            return img
    return None


def evaluate_frcnn_on_split(weights_path, recs, model_name="Faster R-CNN (base)"):
    import time
    from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

    print(f"  Loading {model_name} ...")
    frcnn = fasterrcnn_resnet50_fpn_v2()
    in_feat = frcnn.roi_heads.box_predictor.cls_score.in_features
    frcnn.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes=2)
    frcnn.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    frcnn.to(DEVICE).eval()

    all_gt, all_preds, fps_times = [], [], []

    _img = first_image(recs)
    if _img is not None:
        _w = (
            torch.from_numpy(cv2.cvtColor(_img, cv2.COLOR_BGR2RGB))
            .permute(2, 0, 1)
            .float()
            .unsqueeze(0)
            .to(DEVICE)
            / 255.0
        )
        with torch.no_grad():
            warmup(lambda: frcnn(_w))

    for rec in tqdm(recs, desc=f"  {model_name}"):
        img = cv2.imread(str(rec["image_path"]))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = (
            torch.from_numpy(img_rgb).permute(2, 0, 1).float().unsqueeze(0).to(DEVICE)
            / 255.0
        )

        t0 = time.perf_counter()
        with torch.no_grad():
            out = frcnn(tensor)[0]
        _sync()
        fps_times.append(time.perf_counter() - t0)

        preds = []
        for box, score in zip(out["boxes"].cpu().numpy(), out["scores"].cpu().numpy()):
            x1, y1, x2, y2 = box
            preds.append(
                {
                    "label": "pothole",
                    "conf": float(score),
                    "xmin": float(x1),
                    "ymin": float(y1),
                    "xmax": float(x2),
                    "ymax": float(y2),
                }
            )
        all_gt.append(rec["gt_boxes"])
        all_preds.append(preds)

    # torchvision's Faster R-CNN returns detections down to its internal
    # box_score_thresh (~0.05), which is what mAP's full-curve integration
    # and the sweep both want summarize() applies the thresholds itself.
    m = summarize(all_gt, all_preds, fps_times)

    del frcnn
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return m


print(
    "Metric helpers ready (evaluate / sweep_best_conf / summarize / evaluate_frcnn_on_split)"
)


Metric helpers ready (evaluate / sweep_best_conf / summarize / evaluate_frcnn_on_split)


---
##  Hybrid 1 -- YOLOv8m + CBAM Attention
**What changes:** Adds a Convolutional Block Attention Module (CBAM) after YOLOv8's neck (C2f layers). CBAM applies **channel attention** (what features matter) followed by **spatial attention** (where to look). This directly addresses YOLOv8m's false positives by suppressing irrelevant background texture.

In [10]:
# Defines the CBAM module, channel attention followed by spatial attention, wrapped in a zero-initialized residual gate.
import torch
import torch.nn as nn
import torch.nn.functional as F


class ChannelAttention(nn.Module):
    """Squeeze-and-excitation style channel attention."""

    def __init__(self, in_channels, reduction=16):
        super().__init__()
        mid = max(1, in_channels // reduction)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, in_channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    """Spatial attention using avg+max pooling along channel dim."""

    def __init__(self, kernel_size=7):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=pad, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        scale = self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return scale


class CBAM(nn.Module):
    """
    Convolutional Block Attention Module, wrapped in
    a zero-initialised residual gate.
    """

    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(in_channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)
        self.scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        att = x * self.channel_att(x)  # weight channels
        att = att * self.spatial_att(att)  # weight spatial locations
        return x + torch.tanh(self.scale) * (att - x)


print(" CBAM module defined (residual, zero-init gate)")
# Sanity check: must be an EXACT identity at initialisation
dummy = torch.randn(2, 256, 20, 20)
cbam = CBAM(256)
out = cbam(dummy)
init_diff = (out - dummy).abs().max().item()
print(f"   Input shape : {dummy.shape}")
print(f"   Output shape: {out.shape}")
print(f"   init max|out-in| = {init_diff:.8f}   (MUST be 0.0 -- identity at init)")
assert init_diff == 0.0, "CBAM is not identity at init -- residual gate is broken"


 CBAM module defined (residual, zero-init gate)
   Input shape : torch.Size([2, 256, 20, 20])
   Output shape: torch.Size([2, 256, 20, 20])
   init max|out-in| = 0.00000000   (MUST be 0.0 -- identity at init)


In [11]:
# Defines find_detect_module and get_neck_c2f_modules, the introspection helpers that locate the three neck C2f layers the attention modules attach to.


def find_detect_module(model):
    # Walk model tree to find the Detect layer by class name.
    for name, module in model.named_modules():
        if type(module).__name__ == "Detect":
            return name, module
    return None, None


def get_neck_c2f_modules(model):
    """
    Find the last 3 C2f (or C2fAttn) modules in the neck -- these output
    P3/P4/P5 feature maps that feed into the Detect head.
    Works by name-matching since layer indices shift between versions.
    """
    c2f_layers = []
    for name, module in model.named_modules():
        cname = type(module).__name__
        if cname in ("C2f", "C2fAttn", "C2", "BottleneckCSP"):
            c2f_layers.append((name, module))
    # The last 3 C2f blocks in YOLOv8 neck are P3/P4/P5 outputs
    return c2f_layers[-3:] if len(c2f_layers) >= 3 else c2f_layers


print("find_detect_module + get_neck_c2f_modules defined")


find_detect_module + get_neck_c2f_modules defined


In [12]:
# Defines CBAMHookInjector, which probes each neck C2f layer for its real output channel count and attaches a CBAM forward hook to it.
import torch.nn as nn
import torch


class CBAMHookInjector:
    def __init__(self, yolo_model_nn):
        self._hooks = []
        self.cbams = nn.ModuleList()
        device = next(yolo_model_nn.parameters()).device
        neck_layers = get_neck_c2f_modules(yolo_model_nn)

        print(f"   Found {len(neck_layers)} neck C2f layers to hook:")

        real_channels = [None] * len(neck_layers)
        probe_hooks = []

        for i, (name_, layer) in enumerate(neck_layers):

            def make_probe(idx):
                def fn(mod, inp, out):
                    o = out[0] if isinstance(out, (list, tuple)) else out
                    real_channels[idx] = o.shape[1]

                return fn

            probe_hooks.append(layer.register_forward_hook(make_probe(i)))

        dummy = torch.zeros(1, 3, 64, 64).to(device)
        with torch.no_grad():
            try:
                yolo_model_nn(dummy)
            except:
                pass

        for h in probe_hooks:
            h.remove()

        for i, (name_, layer) in enumerate(neck_layers):
            ch = real_channels[i]
            if ch is None:
                raise RuntimeError(f"Could not probe channels for {name_}")

            print(
                f"     [{i}] {name_}: {type(layer).__name__}  real_output_channels={ch}"
            )

            cbam_mod = CBAM(ch).to(device)
            self.cbams.append(cbam_mod)

            def make_hook(cb):
                def hook_fn(module, inp, out):
                    if isinstance(out, (list, tuple)):
                        return (cb(out[0]),) + out[1:]
                    return cb(out)

                return hook_fn

            h = layer.register_forward_hook(make_hook(cbam_mod))
            self._hooks.append(h)

    def remove(self):
        for h in self._hooks:
            h.remove()
        print("   CBAM hooks removed")


print("CBAMHookInjector defined (attaches CBAM to neck C2f layers via forward hooks)")

CBAMHookInjector defined (attaches CBAM to neck C2f layers via forward hooks)


In [13]:
# Verifies the CBAM injector before training: gates at init leave the output bit-identical, gates forced open change it, confirming both the residual gate and the hook wiring.
import torch
from ultralytics import YOLO


def _set_gates(attn_modules, value):
    """Set every residual gate scalar in the ModuleList to `value`."""
    n = 0
    with torch.no_grad():
        for name, p in attn_modules.named_parameters():
            if name.endswith("scale"):
                p.fill_(value)
                n += 1
    return n


print("=" * 60)
print("  Checking YOLOv8m + CBAM ...")
print("=" * 60)

model = YOLO(YOLOV8M_WEIGHTS)
device = next(model.model.parameters()).device
neck_layers = get_neck_c2f_modules(model.model)
dummy = torch.zeros(1, 3, 64, 64).to(device)


def _capture(layers, tag):
    """Run one forward pass, returning {layer_idx: output tensor}."""
    store, probes = {}, []
    for i, (n, layer) in enumerate(layers):

        def make_p(idx):
            def fn(m, inp, out):
                o = out[0] if isinstance(out, (list, tuple)) else out
                store[idx] = o.detach().clone()

            return fn

        probes.append(layer.register_forward_hook(make_p(i)))
    with torch.no_grad():
        try:
            model.model(dummy)
        except Exception:
            pass
    for h in probes:
        h.remove()
    return store


outputs_before = _capture(neck_layers, "before")

# Attach CBAM hooks
print("\n  Attaching CBAM hooks ...")
try:
    injector = CBAMHookInjector(model.model)
except Exception as e:
    print(f"  FAILED to attach CBAM: {e}")
    raise

# Gates at init must be an EXACT identity
outputs_init = _capture(neck_layers, "init")

# Gates forced open must change
n_gates = _set_gates(injector.cbams, 1.0)
outputs_open = _capture(neck_layers, "open")
_set_gates(injector.cbams, 0.0)  # restore
injector.remove()

# Compare
print(f"\n  Forced {n_gates} residual gate(s) open for the wiring test.")
print("\n  Layer-by-layer results:")
all_ok = True
for i in range(len(neck_layers)):
    before, init, opened = (
        outputs_before.get(i),
        outputs_init.get(i),
        outputs_open.get(i),
    )
    if before is None or init is None or opened is None:
        print(f"  Layer {i}:   no output captured")
        all_ok = False
        continue

    d_init = (before - init).abs().max().item()
    d_open = (before - opened).abs().mean().item()

    ok_a = d_init == 0.0
    ok_b = d_open > 1e-6
    print(
        f"  Layer {i}: [A] identity@init max|d|={d_init:.8f} "
        f"{'PASS' if ok_a else 'FAIL: not identity, residual gate broken'}   "
        f"[B] gate-open mean|d|={d_open:.4f} "
        f"{'PASS: hooks wired' if ok_b else 'FAIL: no change, hook not firing'}"
    )
    all_ok = all_ok and ok_a and ok_b


print("=" * 60)
print(
    "  CBAM verified: identity at init and correctly wired"
    if all_ok
    else "  CBAM CHECK FAILED: do not train"
)
print("=" * 60)
assert all_ok, "CBAM layer check failed"


  Checking YOLOv8m + CBAM ...

  Attaching CBAM hooks ...
   Found 3 neck C2f layers to hook:
     [0] model.15: C2f  real_output_channels=192
     [1] model.18: C2f  real_output_channels=384
     [2] model.21: C2f  real_output_channels=576
   CBAM hooks removed

  Forced 3 residual gate(s) open for the wiring test.

  Layer-by-layer results:
  Layer 0: [A] identity@init max|d|=0.00000000 PASS   [B] gate-open mean|d|=0.1604 PASS: hooks wired
  Layer 1: [A] identity@init max|d|=0.00000000 PASS   [B] gate-open mean|d|=0.2273 PASS: hooks wired
  Layer 2: [A] identity@init max|d|=0.00000000 PASS   [B] gate-open mean|d|=0.2645 PASS: hooks wired
  CBAM verified: identity at init and correctly wired
